# ARRIBA fusion analysis (18 cores, 110 GB RAM)

Process existing FASTQs (multiple lanes per sample) and call fusions with ARRIBA/STAR. Tuned for an 18-core host with ~110 GB RAM and local reference cache under `data/arriba_refs/`.

## Environment

Use the conda/micromamba spec in `env/arriba.yml` and register the kernel:

```bash
micromamba env create -f env/arriba.yml
micromamba activate arriba  # or: conda activate arriba
python -m ipykernel install --user --name arriba --display-name "arriba"
```

Set `RUN_COMMANDS=False` to dry-run cells without executing heavy tools.

In [2]:
from pathlib import Path
import json
import shutil
import subprocess
from datetime import datetime

PROJECT_ROOT = Path("/datos/migccl/exposoma_fusion")
DATA_DIR = PROJECT_ROOT / "fastqs"
REF_DIR = DATA_DIR / "arriba_refs"
RESULTS_DIR = DATA_DIR / "results"
LOG_DIR = RESULTS_DIR / "logs"

THREADS = 16  # leave ~2 cores free on an 18-core host
STAR_SORT_RAM = 90 * 1024 ** 3  # allocate up to 90 GiB for BAM sorting
RUN_COMMANDS = True
TIMESTAMP = datetime.utcnow().strftime("%Y%m%dT%H%M%SZ")

for path in (REF_DIR, RESULTS_DIR, LOG_DIR):
    path.mkdir(parents=True, exist_ok=True)

print(f"Project root: {PROJECT_ROOT}")
print(f"Results dir: {RESULTS_DIR}")
print(f"Notebook timestamp: {TIMESTAMP}")

Project root: /datos/migccl/exposoma_fusion
Results dir: /datos/migccl/exposoma_fusion/fastqs/results
Notebook timestamp: 20251211T210841Z


In [24]:
DATA_DIR

PosixPath('/datos/migccl/exposoma_fusion/fastqs')

In [4]:
import hashlib
from itertools import islice
import gzip


def md5sum(path):
    path = Path(path)
    h = hashlib.md5()
    with open(path, 'rb') as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b''):
            h.update(chunk)
    return h.hexdigest()


def human_gb(num_bytes):
    return num_bytes / (1024 ** 3)


def require_free_space(path, min_gb):
    free = shutil.disk_usage(path).free
    print(f"Free space on {path}: {human_gb(free):.1f} GB (need >= {min_gb} GB)")
    if free < min_gb * 1024 ** 3:
        raise RuntimeError(f"Insufficient disk; free up space or point to a volume with >= {min_gb} GB")


def run_cmd(cmd, cwd=None, env=None, log_file=None, check=True):
    printable = ' '.join(cmd) if isinstance(cmd, (list, tuple)) else cmd
    print(f"$ {printable}")
    if not RUN_COMMANDS:
        print("  (dry-run mode, command not executed)")
        return None
    result = subprocess.run(cmd, cwd=cwd, env=env, check=check)
    if log_file:
        log_file.parent.mkdir(parents=True, exist_ok=True)
        with open(log_file, 'a', encoding='utf-8') as handle:
            handle.write(f"[{datetime.utcnow().isoformat()}] {printable}\n")
    return result


def ensure_uncompressed(src_gz, dest, min_bytes=None):
    src_gz = Path(src_gz)
    dest = Path(dest)
    if dest.exists():
        if min_bytes and dest.stat().st_size < min_bytes:
            print(f"! {dest.name} exists but is too small; removing")
            dest.unlink()
        else:
            print(f"✔ {dest.name} already exists")
            return dest
    cmd = f"set -euo pipefail && gunzip -c '{src_gz}' > '{dest}'"
    run_cmd(['bash', '-lc', cmd])
    if min_bytes and dest.stat().st_size < min_bytes:
        raise RuntimeError(f"{dest.name} looks truncated after decompression (size={human_gb(dest.stat().st_size):.1f} GB)")
    return dest


def remove_path(path):
    path = Path(path)
    if not path.exists():
        return
    if path.is_dir():
        shutil.rmtree(path)
    else:
        path.unlink()


def is_star_index_complete(index_dir):
    required = ['SA', 'Genome', 'genomeParameters.txt', 'chrName.txt', 'chrLength.txt']
    return index_dir.exists() and all((index_dir / name).exists() for name in required)


print("Helpers ready")

Helpers ready


## Sample FASTQs

FASTQs already live on disk with lane-split files like `*_R1_001.fastq.gz`/`*_R2_001.fastq.gz`. Point to the parent directory that holds all sample folders (e.g., `/datos/migccl/exposoma_fusion`) and pick one `SAMPLE_ID` to process.

In [5]:
# Configure where the FASTQs live and which sample to process
FASTQ_BASE = Path("/datos/migccl/exposoma_fusion/fastqs")  # update if stored elsewhere
SAMPLE_ID = "103-MO-15-563486477"  # change to any folder name listed under FASTQ_BASE

sample_fastq_dir = FASTQ_BASE / SAMPLE_ID
if not sample_fastq_dir.exists():
    raise FileNotFoundError(f"Sample dir missing: {sample_fastq_dir}")

r1_files = sorted(sample_fastq_dir.glob("*_R1_*.fastq.gz"))
r2_files = sorted(sample_fastq_dir.glob("*_R2_*.fastq.gz"))
if not r1_files or not r2_files:
    raise RuntimeError("No FASTQs found; check FASTQ_BASE and SAMPLE_ID")
if len(r1_files) != len(r2_files):
    raise RuntimeError(f"Mismatched R1/R2 counts ({len(r1_files)} vs {len(r2_files)})")

# Persist sample metadata for traceability
SAMPLE = {
    "sample_id": SAMPLE_ID,
    "fastq_dir": str(sample_fastq_dir),
    "lanes": len(r1_files),
    "created": TIMESTAMP,
}

sample_result_dir = RESULTS_DIR / SAMPLE_ID
sample_result_dir.mkdir(parents=True, exist_ok=True)
(sample_result_dir / "metadata.json").write_text(json.dumps(SAMPLE, indent=2))

print(f"Sample: {SAMPLE_ID} ({len(r1_files)} lanes)")
print("R1 files:")
for p in r1_files:
    print(f"  - {p}")
print("R2 files:")
for p in r2_files:
    print(f"  - {p}")

Sample: 103-MO-15-563486477 (2 lanes)
R1 files:
  - /datos/migccl/exposoma_fusion/fastqs/103-MO-15-563486477/103-MO-15_S1_L001_R1_001.fastq.gz
  - /datos/migccl/exposoma_fusion/fastqs/103-MO-15-563486477/103-MO-15_S1_L002_R1_001.fastq.gz
R2 files:
  - /datos/migccl/exposoma_fusion/fastqs/103-MO-15-563486477/103-MO-15_S1_L001_R2_001.fastq.gz
  - /datos/migccl/exposoma_fusion/fastqs/103-MO-15-563486477/103-MO-15_S1_L002_R2_001.fastq.gz


## Read QC (FastQC & MultiQC)

Runs FastQC across all lanes and aggregates reports with MultiQC.

In [6]:
qc_dir = sample_result_dir / "qc"
qc_dir.mkdir(exist_ok=True)

fastqc_cmd = [
    "fastqc",
    "-o", str(qc_dir),
    "-t", str(THREADS),
    *(str(p) for p in r1_files + r2_files),
]
run_cmd(fastqc_cmd)
run_cmd(["multiqc", str(qc_dir), "-o", str(qc_dir)])
print(f"FastQC/MultiQC output: {qc_dir}")

$ fastqc -o /datos/migccl/exposoma_fusion/fastqs/results/103-MO-15-563486477/qc -t 16 /datos/migccl/exposoma_fusion/fastqs/103-MO-15-563486477/103-MO-15_S1_L001_R1_001.fastq.gz /datos/migccl/exposoma_fusion/fastqs/103-MO-15-563486477/103-MO-15_S1_L002_R1_001.fastq.gz /datos/migccl/exposoma_fusion/fastqs/103-MO-15-563486477/103-MO-15_S1_L001_R2_001.fastq.gz /datos/migccl/exposoma_fusion/fastqs/103-MO-15-563486477/103-MO-15_S1_L002_R2_001.fastq.gz
application/gzip
application/gzip
application/gzip
application/gzip


Started analysis of 103-MO-15_S1_L001_R1_001.fastq.gz
Started analysis of 103-MO-15_S1_L002_R1_001.fastq.gz
Started analysis of 103-MO-15_S1_L001_R2_001.fastq.gz
Started analysis of 103-MO-15_S1_L002_R2_001.fastq.gz
Approx 5% complete for 103-MO-15_S1_L001_R1_001.fastq.gz
Approx 5% complete for 103-MO-15_S1_L002_R1_001.fastq.gz
Approx 5% complete for 103-MO-15_S1_L001_R2_001.fastq.gz
Approx 5% complete for 103-MO-15_S1_L002_R2_001.fastq.gz
Approx 10% complete for 103-MO-15_S1_L001_R1_001.fastq.gz
Approx 10% complete for 103-MO-15_S1_L002_R1_001.fastq.gz
Approx 10% complete for 103-MO-15_S1_L001_R2_001.fastq.gz
Approx 10% complete for 103-MO-15_S1_L002_R2_001.fastq.gz
Approx 15% complete for 103-MO-15_S1_L001_R1_001.fastq.gz
Approx 15% complete for 103-MO-15_S1_L002_R1_001.fastq.gz
Approx 15% complete for 103-MO-15_S1_L001_R2_001.fastq.gz
Approx 15% complete for 103-MO-15_S1_L002_R2_001.fastq.gz
Approx 20% complete for 103-MO-15_S1_L002_R1_001.fastq.gz
Approx 20% complete for 103-MO-15_

Analysis complete for 103-MO-15_S1_L002_R1_001.fastq.gz


Approx 95% complete for 103-MO-15_S1_L001_R2_001.fastq.gz
Approx 95% complete for 103-MO-15_S1_L002_R2_001.fastq.gz


Analysis complete for 103-MO-15_S1_L001_R1_001.fastq.gz
Analysis complete for 103-MO-15_S1_L001_R2_001.fastq.gz
Analysis complete for 103-MO-15_S1_L002_R2_001.fastq.gz
$ multiqc /datos/migccl/exposoma_fusion/fastqs/results/103-MO-15-563486477/qc -o /datos/migccl/exposoma_fusion/fastqs/results/103-MO-15-563486477/qc



/// ]8;id=216281;https://multiqc.info\MultiQC]8;;\ 🔍 v1.33

       file_search | Search path: /datos/migccl/exposoma_fusion/fastqs/results/103-MO-15-563486477/qc
         searching | ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100% 8/8  tml  
            fastqc | Found 4 reports
     write_results | Data        : /datos/migccl/exposoma_fusion/fastqs/results/103-MO-15-563486477/qc/multiqc_data
     write_results | Report      : /datos/migccl/exposoma_fusion/fastqs/results/103-MO-15-563486477/qc/multiqc_report.html
           multiqc | MultiQC complete


FastQC/MultiQC output: /datos/migccl/exposoma_fusion/fastqs/results/103-MO-15-563486477/qc


## Reference data

Fetch GRCh38 primary assembly, Gencode v41 annotation, ARRIBA annotations, and build a STAR index sized for 150 bp reads. With 110 GB RAM we use higher RAM/thread settings than the lightweight notebook.

In [13]:
REF_DIR

PosixPath('/datos/migccl/exposoma_fusion/fastqs/arriba_refs')

In [14]:
import tarfile

GENOME_FASTA_GZ = REF_DIR / "GRCh38.primary_assembly.genome.fa.gz"
GENOME_FASTA = REF_DIR / "GRCh38.primary_assembly.genome.fa"
GTF_GZ = REF_DIR / "gencode.v41.basic.annotation.gtf.gz"
GTF = REF_DIR / "gencode.v41.basic.annotation.gtf"
STAR_INDEX_DIR = REF_DIR / "star_grch38_primary"
STAR_TARBALL = REF_DIR / "star_grch38_primary.tar.zst"
ARRIBA_TARBALL = REF_DIR / "arriba_v2.5.1.tar.gz"
ARRIBA_DIR = REF_DIR / "arriba_v2.5.1"

FASTA_GZ_MIN_BYTES = 750 * 1024 ** 2
FASTA_MIN_BYTES = 3_000_000_000
MIN_FREE_GB = 80
GTF_MD5 = "4fbcba27425a6411b4b1a1f846551878"
ARRIBA_URL = "https://github.com/suhrig/arriba/releases/download/v2.5.1/arriba_v2.5.1.tar.gz"

require_free_space(REF_DIR, MIN_FREE_GB)

if GENOME_FASTA_GZ.exists() and GENOME_FASTA_GZ.stat().st_size < FASTA_GZ_MIN_BYTES:
    print("! Genome FASTA gzip looks incomplete; removing for fresh download")
    remove_path(GENOME_FASTA_GZ)

if not GENOME_FASTA_GZ.exists():
    run_cmd([
        "curl", "-L", "-o", str(GENOME_FASTA_GZ),
        "https://ftp.ebi.ac.uk/pub/databases/gencode/Gencode_human/release_41/GRCh38.primary_assembly.genome.fa.gz",
    ])
else:
    print("✔ Genome FASTA gzip present")

if GTF_GZ.exists():
    current = md5sum(GTF_GZ)
    if current != GTF_MD5:
        print(f"! GTF md5 mismatch ({current} vs expected {GTF_MD5}); removing")
        remove_path(GTF_GZ)

if not GTF_GZ.exists():
    run_cmd([
        "curl", "-L", "-o", str(GTF_GZ),
        "https://ftp.ebi.ac.uk/pub/databases/gencode/Gencode_human/release_41/gencode.v41.basic.annotation.gtf.gz",
    ])
else:
    print("✔ GTF gzip present")


def tarball_valid(path):
    path = Path(path)
    try:
        with tarfile.open(path, 'r:gz') as tar:
            tar.getmembers()
        return True
    except Exception as exc:  # noqa: BLE001 - log and retry
        print(f"! Tarball validation failed: {exc}")
        return False


if ARRIBA_TARBALL.exists() and not tarball_valid(ARRIBA_TARBALL):
    print("! Arriba tarball corrupted; removing for re-download")
    remove_path(ARRIBA_TARBALL)

if not ARRIBA_TARBALL.exists():
    run_cmd(["curl", "-L", "-o", str(ARRIBA_TARBALL), ARRIBA_URL])
else:
    print("✔ Arriba tarball present")

ensure_uncompressed(GENOME_FASTA_GZ, GENOME_FASTA, min_bytes=FASTA_MIN_BYTES)
ensure_uncompressed(GTF_GZ, GTF)

if ARRIBA_DIR.exists():
    print("✔ ARRIBA annotation directory present")
else:
    run_cmd(["tar", "-xzf", str(ARRIBA_TARBALL), "-C", str(REF_DIR)])

if STAR_INDEX_DIR.exists() and not is_star_index_complete(STAR_INDEX_DIR):
    print("! STAR index exists but is incomplete; removing before rebuild")
    remove_path(STAR_INDEX_DIR)

if not STAR_INDEX_DIR.exists():
    STAR_INDEX_DIR.mkdir(parents=True, exist_ok=True)

if not is_star_index_complete(STAR_INDEX_DIR):
    star_cmd = [
        "STAR",
        "--runMode", "genomeGenerate",
        "--runThreadN", str(THREADS),
        "--genomeDir", str(STAR_INDEX_DIR),
        "--genomeFastaFiles", str(GENOME_FASTA),
        "--sjdbGTFfile", str(GTF),
        "--sjdbOverhang", "149",
        "--limitGenomeGenerateRAM", str(int(100 * 1024 ** 3)),
        "--genomeSAindexNbases", "14",
    ]
    run_cmd(star_cmd, log_file=LOG_DIR / "star_index.log")
else:
    print("✔ STAR index already present and complete")

if is_star_index_complete(STAR_INDEX_DIR):
    if not STAR_TARBALL.exists():
        cmd = f"cd '{REF_DIR}' && tar --use-compress-program='zstd -T0 -19' -cf '{STAR_TARBALL.name}' '{STAR_INDEX_DIR.name}'"
        run_cmd(["bash", "-lc", cmd])
    else:
        print("✔ STAR tarball already cached")
else:
    print("Skipping STAR tarball creation until STAR index is complete")

ARRIBA_FILES = {
    "blacklist": ARRIBA_DIR / "database" / "blacklist_hg38_GRCh38_v2.5.1.tsv.gz",
    "known_fusions": ARRIBA_DIR / "database" / "known_fusions_hg38_GRCh38_v2.5.1.tsv.gz",
    "protein_domains": ARRIBA_DIR / "database" / "protein_domains_hg38_GRCh38_v2.5.1.gff3",
}
ARRIBA_FILES

Free space on /datos/migccl/exposoma_fusion/fastqs/arriba_refs: 111311.2 GB (need >= 80 GB)
✔ Genome FASTA gzip present
✔ GTF gzip present
$ curl -L -o /datos/migccl/exposoma_fusion/fastqs/arriba_refs/arriba_v2.5.1.tar.gz https://github.com/suhrig/arriba/releases/download/v2.5.1/arriba_v2.5.1.tar.gz


  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
  0     0   0     0   0     0     0     0  --:--:-- --:--:-- --:--:--     0
  0     0   0     0   0     0     0     0  --:--:-- --:--:-- --:--:--     0
100 465.6M 100 465.6M   0     0 60169k     0   0:00:07  0:00:07 --:--:-- 66361k
100 465.6M 100 465.6M   0     0 60169k     0   0:00:07  0:00:07 --:--:-- 66361k


✔ GRCh38.primary_assembly.genome.fa already exists
✔ gencode.v41.basic.annotation.gtf already exists
$ tar -xzf /datos/migccl/exposoma_fusion/fastqs/arriba_refs/arriba_v2.5.1.tar.gz -C /datos/migccl/exposoma_fusion/fastqs/arriba_refs
✔ STAR index already present and complete
✔ STAR tarball already cached
✔ STAR index already present and complete
✔ STAR tarball already cached


{'blacklist': PosixPath('/datos/migccl/exposoma_fusion/fastqs/arriba_refs/arriba_v2.5.1/database/blacklist_hg38_GRCh38_v2.5.1.tsv.gz'),
 'known_fusions': PosixPath('/datos/migccl/exposoma_fusion/fastqs/arriba_refs/arriba_v2.5.1/database/known_fusions_hg38_GRCh38_v2.5.1.tsv.gz'),
 'protein_domains': PosixPath('/datos/migccl/exposoma_fusion/fastqs/arriba_refs/arriba_v2.5.1/database/protein_domains_hg38_GRCh38_v2.5.1.gff3')}

## Alignment (STAR) and fusion calling (ARRIBA)

Reads from all lanes are fed directly to STAR via comma-separated lists. ARRIBA consumes the sorted BAM.

In [15]:
star_out_dir = sample_result_dir / "star"
star_out_dir.mkdir(parents=True, exist_ok=True)

# Define outputs
arriba_out = sample_result_dir / "arriba_fusions.tsv"
arriba_discarded = sample_result_dir / "arriba_fusions.discarded.tsv"
star_prefix = star_out_dir / SAMPLE_ID

read_files_r1 = ",".join(str(p) for p in r1_files)
read_files_r2 = ",".join(str(p) for p in r2_files)

# Construct the pipeline command (STAR | Arriba)
# We use a single shell string passed to 'bash -c' to handle the pipe.
# STAR parameters are tuned for performance (piping to stdout) and sensitivity (Arriba recommendations).

pipeline_cmd = (
    f"STAR --runThreadN {THREADS} "
    f"--genomeDir {STAR_INDEX_DIR} --genomeLoad NoSharedMemory "
    f"--readFilesIn {read_files_r1} {read_files_r2} --readFilesCommand zcat "
    f"--outStd BAM_Unsorted --outSAMtype BAM Unsorted --outSAMunmapped Within --outBAMcompression 0 "
    f"--outFilterMultimapNmax 50 --peOverlapNbasesMin 10 --alignSplicedMateMapLminOverLmate 0.5 --alignSJstitchMismatchNmax 5 -1 5 5 "
    f"--chimSegmentMin 10 --chimOutType WithinBAM HardClip --chimJunctionOverhangMin 10 --chimScoreDropMax 30 "
    f"--chimScoreJunctionNonGTAG 0 --chimScoreSeparation 1 --chimSegmentReadGapMax 3 --chimMultimapNmax 50 "
    f"--outFileNamePrefix {star_prefix}. "
    f"| "
    f"arriba -x /dev/stdin "
    f"-o {arriba_out} -O {arriba_discarded} "
    f"-a {GENOME_FASTA} -g {GTF} "
    f"-b {ARRIBA_FILES['blacklist']} -k {ARRIBA_FILES['known_fusions']} -t {ARRIBA_FILES['known_fusions']} -p {ARRIBA_FILES['protein_domains']}"
)

run_cmd(['bash', '-c', pipeline_cmd], log_file=LOG_DIR / f"pipeline_{SAMPLE_ID}.log")

print(f"Pipeline finished.")
print(f"Fusions: {arriba_out}")

# Note: This pipeline does NOT produce a sorted BAM file on disk.
aligned_bam = None

$ bash -c STAR --runThreadN 16 --genomeDir /datos/migccl/exposoma_fusion/fastqs/arriba_refs/star_grch38_primary --genomeLoad NoSharedMemory --readFilesIn /datos/migccl/exposoma_fusion/fastqs/103-MO-15-563486477/103-MO-15_S1_L001_R1_001.fastq.gz,/datos/migccl/exposoma_fusion/fastqs/103-MO-15-563486477/103-MO-15_S1_L002_R1_001.fastq.gz /datos/migccl/exposoma_fusion/fastqs/103-MO-15-563486477/103-MO-15_S1_L001_R2_001.fastq.gz,/datos/migccl/exposoma_fusion/fastqs/103-MO-15-563486477/103-MO-15_S1_L002_R2_001.fastq.gz --readFilesCommand zcat --outStd BAM_Unsorted --outSAMtype BAM Unsorted --outSAMunmapped Within --outBAMcompression 0 --outFilterMultimapNmax 50 --peOverlapNbasesMin 10 --alignSplicedMateMapLminOverLmate 0.5 --alignSJstitchMismatchNmax 5 -1 5 5 --chimSegmentMin 10 --chimOutType WithinBAM HardClip --chimJunctionOverhangMin 10 --chimScoreDropMax 30 --chimScoreJunctionNonGTAG 0 --chimScoreSeparation 1 --chimSegmentReadGapMax 3 --chimMultimapNmax 50 --outFileNamePrefix /datos/migcc

[2025-12-12T12:37:21] Loading annotation from '/datos/migccl/exposoma_fusion/fastqs/arriba_refs/gencode.v41.basic.annotation.gtf' 
[2025-12-12T12:37:28] Reading chimeric alignments from '/dev/stdin' [2025-12-12T12:37:28] Reading chimeric alignments from '/dev/stdin' (total=6963347)
[2025-12-12T12:53:25] Marking multi-mapping alignments (total=6963347)
[2025-12-12T12:53:25] Marking multi-mapping alignments 

(marked=5622862)
[2025-12-12T12:53:28] Detecting strandedness (reverse)
[2025-12-12T12:53:28] Assigning strands to alignments 
[2025-12-12T12:53:29] Annotating alignments 
[2025-12-12T12:53:29] Annotating alignments 
[2025-12-12T12:53:49] Filtering duplicates [2025-12-12T12:53:49] Filtering duplicates (remaining=3235455)
[2025-12-12T12:53:55] Filtering mates which do not map to interesting contigs (1 2 3 4 5 6 7 8 9 10 11 12 13 14 15 16 17 18 19 20 21 22 X Y AC_* NC_*) (remaining=3235455)
[2025-12-12T12:53:55] Filtering mates which do not map to interesting contigs (1 2 3 4 5 6 7 8 9 10 11 12 13 14 15 16 17 18 19 20 21 22 X Y AC_* NC_*) (remaining=3004674)
[2025-12-12T12:53:56] Filtering mates which only map to viral contigs (AC_* NC_*) (remaining=3004674)
[2025-12-12T12:53:56] Filtering mates which only map to viral contigs (AC_* NC_*) (remaining=3004674)
[2025-12-12T12:53:57] Filtering viral contigs with expression lower than the top 5 (remaining=3004674)
[2025-12-12T12:53:57] Filter

197174)
[2025-12-12T12:54:43] Merging adjacent fusion breakpoints (remaining=196870)
[2025-12-12T12:54:45] Filtering multi-mapping fusions by alignment score and read support (remaining=196870)
[2025-12-12T12:54:45] Filtering multi-mapping fusions by alignment score and read support (remaining=122815)
[2025-12-12T12:55:15] Estimating expected number of fusions by random chance (e-value) 
(remaining=122815)
[2025-12-12T12:55:15] Estimating expected number of fusions by random chance (e-value) 
[2025-12-12T12:55:21] Filtering fusions with both breakpoints in adjacent non-coding/intergenic regions [2025-12-12T12:55:21] Filtering fusions with both breakpoints in adjacent non-coding/intergenic regions (remaining=115974)
[2025-12-12T12:55:22] Filtering intragenic fusions with both breakpoints in exonic regions (remaining=115974)
[2025-12-12T12:55:22] Filtering intragenic fusions with both breakpoints in exonic regions (remaining=104927)
[2025-12-12T12:55:22] Filtering fusions with <2 support

[2025-12-12T12:56:24] Writing fusions to file '/datos/migccl/exposoma_fusion/fastqs/results/103-MO-15-563486477/arriba_fusions.tsv' 
[2025-12-12T12:56:25] Writing discarded fusions to file '/datos/migccl/exposoma_fusion/fastqs/results/103-MO-15-563486477/arriba_fusions.discarded.tsv'
[2025-12-12T12:56:25] Writing discarded fusions to file '/datos/migccl/exposoma_fusion/fastqs/results/103-MO-15-563486477/arriba_fusions.discarded.tsv'
[2025-12-12T12:57:12] Freeing resources
[2025-12-12T12:57:12] Freeing resources
[2025-12-12T12:57:23] Done (elapsed time=00:20:44, CPU time=00:08:47, peak memory=9.8gb)
[2025-12-12T12:57:23] Done (elapsed time=00:20:44, CPU time=00:08:47, peak memory=9.8gb)
Pipeline finished.
Fusions: /datos/migccl/exposoma_fusion/fastqs/results/103-MO-15-563486477/arriba_fusions.tsv
Pipeline finished.
Fusions: /datos/migccl/exposoma_fusion/fastqs/results/103-MO-15-563486477/arriba_fusions.tsv


In [16]:
arriba_plot_dir = sample_result_dir / "arriba_plots"
arriba_plot_dir.mkdir(exist_ok=True)

# Visualization (draw_fusions.R)
# This step requires a sorted and indexed BAM file.
# Since the high-performance pipeline above does not save the BAM, we skip plotting.

cytobands = ARRIBA_DIR / "database" / "cytobands_hg38_GRCh38_v2.5.1.tsv"
draw_script = ARRIBA_DIR / "draw_fusions.R"

if aligned_bam and aligned_bam.exists() and draw_script.exists() and cytobands.exists():
    draw_cmd = [
        str(draw_script),
        "--fusions=" + str(arriba_out),
        "--alignments=" + str(aligned_bam),
        "--output=" + str(arriba_plot_dir / "arriba_plot.pdf"),
        "--annotation=" + str(GTF),
        "--cytobands=" + str(cytobands),
        "--proteinDomains=" + str(ARRIBA_FILES["protein_domains"]),
    ]
    run_cmd(draw_cmd, log_file=LOG_DIR / f"draw_fusions_{SAMPLE_ID}.log")
else:
    draw_cmd = [
        str(draw_script),
        "--fusions=" + str(arriba_out),
        # "--alignments=" + str(aligned_bam),
        "--output=" + str(arriba_plot_dir / "arriba_plot.pdf"),
        "--annotation=" + str(GTF),
        # "--cytobands=" + str(cytobands),
        "--proteinDomains=" + str(ARRIBA_FILES["protein_domains"]),
    ]
    run_cmd(draw_cmd, log_file=LOG_DIR / f"draw_fusions_{SAMPLE_ID}.log")

arriba_out

$ /datos/migccl/exposoma_fusion/fastqs/arriba_refs/arriba_v2.5.1/draw_fusions.R --fusions=/datos/migccl/exposoma_fusion/fastqs/results/103-MO-15-563486477/arriba_fusions.tsv --output=/datos/migccl/exposoma_fusion/fastqs/results/103-MO-15-563486477/arriba_plots/arriba_plot.pdf --annotation=/datos/migccl/exposoma_fusion/fastqs/arriba_refs/gencode.v41.basic.annotation.gtf --proteinDomains=/datos/migccl/exposoma_fusion/fastqs/arriba_refs/arriba_v2.5.1/database/protein_domains_hg38_GRCh38_v2.5.1.gff3


Loading annotation
Loading annotation
Read 1968859 records
Read 1968859 records
Loading protein domains
Loading protein domains
Read 134356 records
Read 134356 records
Drawing fusion #1: TCF3:PBX1
Drawing fusion #1: TCF3:PBX1
Drawing fusion #2: ENSG00000289315(38979),MBNL1-AS1(18176):MBNL1
Drawing fusion #2: ENSG00000289315(38979),MBNL1-AS1(18176):MBNL1
Drawing fusion #3: PSPC1:PSPC1-AS2(1939),PSPC1(24677)
Drawing fusion #3: PSPC1:PSPC1-AS2(1939),PSPC1(24677)
Drawing fusion #4: LINC02200(65443),APC(30003):APC
Drawing fusion #4: LINC02200(65443),APC(30003):APC
Drawing fusion #5: ENSG00000287700:DMD
Drawing fusion #5: ENSG00000287700:DMD
Drawing fusion #6: ENSG00000287700:DMD
Drawing fusion #6: ENSG00000287700:DMD
Drawing fusion #7: RAF1:TMEM40
Drawing fusion #7: RAF1:TMEM40
Drawing fusion #8: RAF1:TMEM40
Drawing fusion #8: RAF1:TMEM40
Drawing fusion #9: LINC02227:ENSG00000253134(38552),LINC02227(3318)
Drawing fusion #9: LINC02227:ENSG00000253134(38552),LINC02227(3318)
Drawing fusion #10

PosixPath('/datos/migccl/exposoma_fusion/fastqs/results/103-MO-15-563486477/arriba_fusions.tsv')

In [17]:
arriba_plot_dir

PosixPath('/datos/migccl/exposoma_fusion/fastqs/results/103-MO-15-563486477/arriba_plots')

## Curated fusion summary

Highlight protocol-relevant lesions (DUX4/IGH, ETV6-RUNX1, TCF3-PBX1/PBX3, CRLF2/JAK/ABL-class).

In [20]:
import pandas as pd

if not arriba_out.exists():
    raise FileNotFoundError(f"{arriba_out} not found; run ARRIBA first")

fusions = pd.read_table(arriba_out)
key_genes = {
    "DUX4", "IGH", "ETV6", "RUNX1", "TCF3", "PBX1", "PBX3",
    "CRLF2", "JAK1", "JAK2", "EPOR", "ABL1", "ABL2", "PDGFRA", "PDGFRB", "CSF1R", "LYN",
}

fusions["protocol_flag"] = fusions.apply(
    lambda row: any(g in key_genes for g in (row["#gene1"], row["gene2"])),
    axis=1,
)
print(f"Detected fusions: {len(fusions)}")
fusions.head(20)

Detected fusions: 33


,#gene1,gene2,strand1(gene/fusion),strand2(gene/fusion),breakpoint1,breakpoint2,site1,site2,type,split_reads1,...,gene_id2,transcript_id1,transcript_id2,direction1,direction2,filters,fusion_transcript,peptide_sequence,read_identifiers,protocol_flag
0,TCF3,PBX1,-/-,+/+,chr19:1619111,chr1:164792494,CDS/splice-site,CDS/splice-site,translocation,80,...,ENSG00000185630.20,ENST00000453954.6,ENST00000420696.7,upstream,upstream,"duplicates(24),mismappers(3),mismatches(1)",CTGGGCGGGCGGCACGCAGGCCTG___GTTGGAGGCAGCCACCCCG...,LGGRHAGLVGGSHPEDGLAGSTSLMHNHAALPSQPGTLPDLSRPPD...,"LH00486:12:223CJ7LT1:1:1106:6890:29728,LH00486...",True
1,"ENSG00000289315(38979),MBNL1-AS1(18176)",MBNL1,./+,+/+,chr3:152244440,chr3:152299405,intergenic,5'UTR/splice-site,deletion/read-through,2,...,ENSG00000152601.18,.,ENST00000282488.11,downstream,upstream,"duplicates(9),mismappers(4)",AAAAAAATAATGTAACTAGAGAAGAGAGTTGAGCAGAGAACTGAGC...,.,"LH00486:12:223CJ7LT1:1:2252:31734:20707,LH0048...",False
2,PSPC1,"PSPC1-AS2(1939),PSPC1(24677)",-/-,./-,chr13:19730239,chr13:19677823,CDS/splice-site,intergenic,deletion/read-through,12,...,.,ENST00000338910.9,.,upstream,downstream,"duplicates(2),low_entropy(1)",CTAAG___ACATGAAGAGGAGCATCGGCGGCGTGAGGAAGAAATGA...,LRHEEEHRRREEEMIRHREQEELRRQQEGFKPNYMEN|gdkrkcg*,"LH00486:12:223CJ7LT1:1:2108:50374:28879,LH0048...",False
3,"LINC02200(65443),APC(30003)",APC,./+,+/+,chr5:112707882,chr5:112754873,intergenic,5'UTR/splice-site,deletion/read-through,6,...,ENSG00000134982.17,.,ENST00000257430.9,downstream,upstream,"duplicates(1),mismappers(1)",TCGGCGCCCCCTATGTACGCCTCCCTGGGCTCGGGTCCGGTCGCCC...,.,"LH00486:12:223CJ7LT1:1:1155:49366:5566,LH00486...",False
4,ENSG00000287700,DMD,-/-,-/-,chrX:33725628,chrX:33020200,exon/splice-site,CDS/splice-site,deletion,4,...,ENSG00000198947.18,ENST00000653485.1,ENST00000357033.9,upstream,downstream,.,GTCTAAGGTTTTTGCAGCGCTTCCCACCCCCATGCACGTGTAAATT...,.,"LH00486:12:223CJ7LT1:1:1120:36366:17487,LH0048...",False
5,ENSG00000287700,DMD,-/-,-/-,chrX:33723958,chrX:33020200,exon/splice-site,CDS/splice-site,deletion,0,...,ENSG00000198947.18,ENST00000653485.1,ENST00000357033.9,upstream,downstream,mismappers(1),GGTTGATGTACGTGACTTCAACAATCATAAT|ATGAAAGAGAAGAT...,.,LH00486:12:223CJ7LT1:1:2122:21406:18256,False
6,RAF1,TMEM40,-/-,-/-,chr3:12590798,chr3:12763836,CDS/splice-site,intron,duplication,2,...,ENSG00000088726.16,ENST00000442415.7,.,upstream,downstream,.,CACGGCATGTGAACATTCTGCTTTTCATGGGGTACATGACAAAGGA...,SLYKHLHVQETKFQMFQLIDIARQTAQGMe|asfsdlgshwgvpis...,"LH00486:12:223CJ7LT1:2:1166:17606:7537,LH00486...",False
7,RAF1,TMEM40,-/-,-/-,chr3:12590798,chr3:12749840,CDS/splice-site,5'UTR/splice-site,duplication,1,...,ENSG00000088726.16,ENST00000442415.7,ENST00000264728.8,upstream,downstream,.,TGGCAATTGTGACCCAGTGGTGCGAGGGCAGCAGCCTCTACAAACA...,AIVTQWCEGSSLYKHLHVQETKFQMFQLIDIARQTAQGMe|kshgd...,"LH00486:12:223CJ7LT1:2:2230:14814:8306,LH00486...",False
8,LINC02227,"ENSG00000253134(38552),LINC02227(3318)",-/+,./+,chr5:158359646,chr5:158317365,intron,intergenic,duplication,11,...,.,.,.,downstream,upstream,"duplicates(9),low_entropy(2),mismatches(6)",GGAAAAATGGATAGAAATAAGAAAATGAATACATGAATGATCTCAG...,.,"LH00486:12:223CJ7LT1:1:1122:1037:8466,LH00486:...",False
9,"ABR(41471),ENSG00000262213(13146)",ABR,./-,-/-,chr17:1228793,chr17:1125367,intergenic,CDS/splice-site,deletion/read-through,6,...,ENSG00000159842.16,.,ENST00000302538.10,upstream,downstream,duplicates(1),GAGACCCCCGGGCGCGAGCACCTGCCCCCGTGGCGGCGCCGCAGGT...,.,"LH00486:12:223CJ7LT1:1:1137:46389:1240,LH00486...",False


In [23]:
fusions.loc[:, ["#gene1", "gene2", "protocol_flag", "confidence"]]

,#gene1,gene2,protocol_flag,confidence
0,TCF3,PBX1,True,high
1,"ENSG00000289315(38979),MBNL1-AS1(18176)",MBNL1,False,high
2,PSPC1,"PSPC1-AS2(1939),PSPC1(24677)",False,high
3,"LINC02200(65443),APC(30003)",APC,False,high
4,ENSG00000287700,DMD,False,high
5,ENSG00000287700,DMD,False,low
6,RAF1,TMEM40,False,high
7,RAF1,TMEM40,False,high
8,LINC02227,"ENSG00000253134(38552),LINC02227(3318)",False,medium
9,"ABR(41471),ENSG00000262213(13146)",ABR,False,medium


In [21]:
fusions[fusions["protocol_flag"]]

,#gene1,gene2,strand1(gene/fusion),strand2(gene/fusion),breakpoint1,breakpoint2,site1,site2,type,split_reads1,...,gene_id2,transcript_id1,transcript_id2,direction1,direction2,filters,fusion_transcript,peptide_sequence,read_identifiers,protocol_flag
0,TCF3,PBX1,-/-,+/+,chr19:1619111,chr1:164792494,CDS/splice-site,CDS/splice-site,translocation,80,...,ENSG00000185630.20,ENST00000453954.6,ENST00000420696.7,upstream,upstream,"duplicates(24),mismappers(3),mismatches(1)",CTGGGCGGGCGGCACGCAGGCCTG___GTTGGAGGCAGCCACCCCG...,LGGRHAGLVGGSHPEDGLAGSTSLMHNHAALPSQPGTLPDLSRPPD...,"LH00486:12:223CJ7LT1:1:1106:6890:29728,LH00486...",True


## Next steps
- Flip `RUN_COMMANDS=False` to smoke-test commands without execution.
- Swap `SAMPLE_ID` to process another folder; results land in `data/results/<sample>/`.
- If references already exist on shared storage, point `REF_DIR` there to avoid downloads.
- Consider wrapping STAR/ARRIBA steps in a job script for batch processing across all folders under `FASTQ_BASE`.